<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_28_minimal_reproducible_pipeline_project.ipynb">9.28 最小可复现处理项目</a></li>
        <li>下一节： <a href="9_x_further_reading_and_workflow.ipynb">9.x 延伸阅读与后续实践方向</a></li>
    </ul>
</div>


## 9.29 源表与检测产品验证：从 PyBDSF、SoFiA 到可发布样本

第 9.15 节已经介绍了连续谱源表生成的基本概念：局部背景、局部 RMS、island、component、source、阈值取舍和残差检查。第 9.24 与 9.28 节又把 PyBDSF、SoFiA、CARTA、Astropy 生态、配置文件和 provenance 放进项目结构。这里继续推进一步：源表、谱线候选表、mask、moment 图和区域文件如何从软件输出变成可发布、可审查、可复现的科学样本。

这个问题在 summer school 和真实研究之间非常关键。教材中常见的处理流程会在“运行源搜索工具，得到源表”处结束；真实项目却必须继续回答：源表的选择函数是什么，哪些候选源只应作为 candidate，哪些检测应转化为上限，边缘伪源如何排除，注入源恢复率如何估计，负源统计如何解释，谱线 mask 是否受通道相关噪声影响，CARTA 中手工画出的 region 如何进入 Python 测量脚本，最终发布表如何记录版本和限制。

因此，本节讨论的不是某个软件按钮，而是检测产品的测量链。PyBDSF 与 SoFiA 是两个典型入口：前者常用于连续谱图像中的二维源搜索，后者常用于谱线数据立方中的三维源搜索和可靠性估计。两者的输出都不能直接等同于科学结论；它们必须和输入图像身份、参数文件、模型图、残差图、mask、区域文件、验证指标和发布规则一起进入 provenance bundle。


### 9.29.1 源表是选择函数，不只是坐标清单

源表最容易被误读为一列坐标、峰值流量和积分流量。这样的表格形式很直观，但它隐藏了最重要的信息：这些源为什么会被检测到，哪些真实源会被漏掉，哪些噪声峰或残余旁瓣可能混入表中。也就是说，源表首先是一个选择函数定义下的数据产品。

设天空中真实源的参数为 $\boldsymbol{\phi}$，包括位置、峰值流量、总流量、角尺寸、谱指数、线宽、速度、形态和环境噪声。源表生成过程可以抽象为

$$
C = \mathcal{D}(I,\sigma,M,\boldsymbol{\theta}),
$$

其中 $I$ 是图像或数据立方，$\sigma$ 是局部噪声估计，$M$ 是有效区域、mask 或主波束限制，$\boldsymbol{\theta}$ 是检测参数，$C$ 是输出目录。目录中的每一行都携带一个隐含条件：该源在给定 $I$、$\sigma$、$M$ 和 $\boldsymbol{\theta}$ 下被检测。若这些条件没有记录，目录就无法用于严格的源计数、交叉匹配、上限统计或样本选择。

选择函数并不只影响弱源。亮而复杂的射电星系可能被拆成多个 component；低面亮度展源可能有高积分通量却低峰值信噪比；主波束边缘源可能因为噪声放大而不可靠；谱线源可能只在某些平滑核下显著。源表发布时必须说明这些分支规则，否则不同项目之间的目录差异会被误解释为天体物理差异。


![检测产品生命周期](figures/practical_catalog_detection_product_lifecycle.png)

图中把检测产品从输入图像或数据立方推进到科学样本。源表的科学含义不是表格本身，而是输入身份、检测参数、附属产品、验证证据和发布规则的组合。模型图、残差图、mask 和验证包应与源表一起保存。


### 9.29.2 连续谱二维检测与谱线三维检测

连续谱源搜索通常面对二维恢复图像。核心统计量是局部信噪比，常写成

$$
\rho(x,y)=\frac{I(x,y)-B(x,y)}{\sigma(x,y)},
$$

其中 $B(x,y)$ 是背景或零点估计，$\sigma(x,y)$ 是局部 RMS。PyBDSF 风格流程先估计背景和 RMS，再识别显著 island，把 island 分解为高斯 component，最后按规则合并为 source。二维检测的难点在于局部噪声不均匀、亮源残差、主波束边缘、分量合并和复杂形态。

谱线源搜索面对三维数据立方，统计问题不同。信号不仅有空间位置，还有速度或频率宽度。SoFiA 风格流程常先对 cube 做多尺度空间和频谱平滑，再在不同平滑核下寻找候选体素集合，生成三维 mask、候选源表、moment 图和可靠性估计。三维检测的难点在于通道相关噪声、平滑核选择、连续谱扣除残差、边缘通道、速度轴定义和可靠性建模。

两者的共同核心是局部噪声、阈值、mask、残差和选择函数；主要差异是检测维度和误差结构。连续谱目录常要解释二维峰值、积分通量和 component/source 合并；谱线目录常要解释三维连通体、谱线宽度、总线通量、可靠性和 completeness。把二者简单写成“运行源搜索”会掩盖这些差异。


![连续谱与谱线源搜索差异](figures/practical_catalog_pybdsf_sofia_comparison_map.png)

图中对比连续谱二维检测和谱线三维检测。连续谱路径强调局部噪声图、island/component/source 层级、源表与残差；谱线路径强调数据立方、空间和频谱平滑、三维 mask、候选源表、moment 图、可靠性和完整性。两者共享检测统计思想，但验证证据不同。


### 9.29.3 阈值、完整性、可靠性与选择函数

阈值改变的不只是源数。较低阈值会提高 completeness，即更容易找回弱源，却会引入更多噪声峰、残余旁瓣和边缘伪源；较高阈值会提高 reliability，却会漏掉弱源、低面亮度源和宽线源。真正的参数选择应由科学目标决定，而不是由工具默认值决定。

连续谱目录常用负源统计作为 reliability 的近似诊断。若对图像取负并用同样参数运行检测，显著负源数量可以估计噪声峰和残差伪源的规模。在理想高斯噪声且没有系统残差时，可用

$$
R(\rho)\simeq 1-\frac{N_-(\rho)}{N_+(\rho)},
$$

其中 $N_+$ 和 $N_-$ 分别是在给定信噪比以上的正源和负源数。这个公式只是近似，因为真实图像中的旁瓣、清洁残差和主波束噪声并不完全对称；但它能暴露低阈值目录的伪检风险。

Completeness 更适合通过注入源恢复估计。把一组已知流量、尺寸和位置的人工源注入图像或 cube，按同一检测流程重新运行，得到

$$
C(S,\theta,r)=\frac{N_{\rm recovered}(S,\theta,r)}{N_{\rm injected}(S,\theta,r)},
$$

其中 $S$ 表示流量或线通量，$\theta$ 表示源尺寸或线宽，$r$ 表示距场中心或主波束响应。这个函数比单个“检测阈值”更接近目录的真实选择函数。谱线项目还需要把通道宽度、线宽、平滑核、mask 生长和频谱相关噪声放进 completeness 估计中。


![阈值与选择函数](figures/practical_catalog_selection_function_thresholds.png)

图中用示意曲线说明阈值与完整性、可靠性和伪检率之间的关系。深阈值区域能提高弱源恢复，但伪检风险高；高阈值区域更可靠，却会漏掉弱源。真实目录需要用注入源恢复、负源统计、边缘排查和人工复核样本共同约束这一取舍。


### 9.29.4 检测产品的来源记录

检测产品的 provenance 至少应覆盖四层。第一层是输入身份：图像或数据立方文件、生成它的成像参数、恢复波束、单位、WCS、频率轴或速度轴、主波束校正状态、有效区域和噪声估计。对连续谱图像，还应说明是否为主波束校正前图像，源搜索是在校正前还是校正后进行；对谱线 cube，还应说明速度定义、频率参考系、通道宽度、连续谱扣除方法和被排除通道。

第二层是参数清单。PyBDSF 工作流应保存背景/RMS box、像素阈值、island 阈值、是否使用 adaptive RMS、是否输出 model image、residual image、island mask、component catalogue 和 source catalogue。SoFiA 工作流应保存平滑核、阈值、reliability 设置、mask 生长、最小体素数、边缘排除、输出 moment 产品和可靠性表。参数必须来自受版本控制的文件，而不是只存在于交互命令历史中。

第三层是附属产品。没有 model image 和 residual image，连续谱源表很难判断过拟合、漏拟合和亮源旁瓣；没有 mask、moment map 和 reliability table，谱线候选表很难判断噪声连通体和真实线发射；没有 CARTA region 文件或等价区域文件，后续 Python 测量无法追溯人工选择。第四层是验证记录，包括注入源恢复、负源统计、边缘排查、交叉匹配、人工复核样本和发布分支规则。


![检测产品来源记录](figures/practical_catalog_provenance_chain.png)

图中把检测产品的来源记录拆成输入身份、参数清单、检测运行、附属产品、验证记录和发布包。源表如果脱离模型、残差、遮罩和验证记录，就很难说明完整性、可靠性和样本限制。


### 9.29.5 发布分支：纳入、复核、候选、上限与排除

检测结果进入科学样本之前，应先经过发布分支。高信噪、残差干净、远离边缘且形态简单的源可以直接进入主目录；高信噪但形态复杂的射电星系需要人工复核和分量合并规则；低信噪但位于干净区域的源可作为候选或用于统计上限；靠近主波束边缘、强源旁瓣、坏通道或 mask 边界的检测应带有质量标记，严重时只能排除；只在强平滑后出现的谱线候选需要特别检查线宽、通道相关噪声和可靠性。

上限也是科学产品。非检测并不等于没有信息。若某个先验位置没有达到检测阈值，仍可根据局部 RMS、预期线宽或波束面积给出上限。上限必须记录测量区域、噪声估计、线宽假设、主波束响应和显著性定义。若只保存检测源而丢弃非检测信息，样本会产生选择偏差。

发布表最好区分主样本、候选样本、排除样本和上限表。主样本用于定量科学结论，候选样本用于后续确认，排除样本用于解释边缘、旁瓣和坏通道规则，上限表用于非检测约束。这样的分支结构看似复杂，却能显著降低样本解释中的歧义。


![源表发布决策矩阵](figures/practical_catalog_release_decision_matrix.png)

图中给出一个发布分支矩阵。检测强度、形态复杂度、残差环境、边缘位置和平滑依赖性共同决定结果是直接纳入、人工复核、保留候选、只给上限还是排除。矩阵不是固定规则，而是提醒发布样本必须保存分支依据。


### 9.29.6 教学案例：连续谱源表、谱线候选与区域测量的闭环

设想一个近邻星系场，同时包含连续谱致密源和中性氢谱线数据立方。连续谱图像需要生成背景源目录，用于交叉匹配和识别可能污染谱线 cube 的连续谱源；谱线 cube 需要搜索气体云和弱外延结构；CARTA 用于交互检查复杂区域和坏通道；Python 脚本用于最终通量、质量和上限测量。

项目可以把检测层写成一个独立配置文件：

```yaml
continuum_catalog:
  input_image: products/images/cont_restored.fits
  search_image: primary_beam_uncorrected
  finder: pybdsf
  rms_box: [80, 20]
  pixel_threshold: 5.0
  island_threshold: 3.0
  output_model: true
  output_residual: true
  edge_pb_limit: 0.3

line_catalog:
  input_cube: products/cubes/hi_contsub_cube.fits
  finder: sofia
  spatial_smoothing: [0, 1, 2]
  spectral_smoothing: [0, 3, 5]
  reliability_threshold: 0.95
  min_channels: 3
  edge_channel_exclusion: 5

regions_and_measurement:
  region_file: validation/carta_regions.reg
  measurement_script: scripts/measure_catalog_products.py
  upper_limit_linewidth: 20 km_per_s
```

这个片段的重点不是参数数值，而是把连续谱、谱线、区域文件和测量脚本放进同一个来源链。连续谱源表输出后，必须检查 model image 和 residual image；若亮源残差污染谱线 cube，应回到连续谱扣除或 flag。谱线候选输出后，必须检查三维 mask、moment 图、可靠性表和通道切片；若候选只在某个平滑核下出现，应进入候选样本而不是主样本。CARTA 中人工画出的区域不能只存在于交互界面，必须保存为 region 文件，并由 Python 测量脚本读取。最终报告需要列出主样本、候选样本、上限表和排除规则。

这个案例能把多个常见教学断点连起来：PyBDSF 不只是连续谱源检测命令，SoFiA 不只是谱线源表生成器，CARTA 不只是看图软件，Python 脚本不只是最终算数工具。它们共同构成从检测到发布的测量链。每个工具都需要参数、附属产品、验证和 provenance 支撑。


### 9.29.7 本节结论

源表和谱线候选表是经过检测统计、参数选择、模型假设和验证规则塑造的数据产品。连续谱二维检测更强调局部 RMS、island/component/source 分层、残差和分量合并；谱线三维检测更强调平滑核、三维 mask、通道相关噪声、可靠性和完整性。PyBDSF、SoFiA、CARTA 和 Python 测量脚本分别处在这条链的不同位置，不能只用软件名概括其科学作用。

一个可发布样本至少应保存输入身份、参数文件、软件版本、命令日志、模型图、残差图、mask、region、可靠性表、注入源恢复、负源统计、边缘排查、交叉匹配和分支规则。这样得到的源表才不只是“检测到了什么”，而是能说明“在什么条件下检测到、哪些会漏掉、哪些可能是伪源、哪些只应作为上限或候选”的科学产品。
